# Credit derivatives (CDS)

**Prerequisites:** **06**. Single-name **`credit_default_swap`** with a **`HazardCurve`**; model key **`hazard_rate`**.


## Concept

- **Premium leg** vs **protection leg**; survival built from **hazard**.
- **CS01** bumps spreads / hazard to summarize credit DV01.
- **Survival** \(S(t)\) links to cumulative hazard — ask for registry metrics that expose it when available.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, HazardCurve, MarketContext

print("Imports OK (CDS).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

cds = json.dumps(
    {
        "type": "credit_default_swap",
        "spec": {
            "id": "CDS-CORP-5Y",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "convention": "isda_na",
            "premium": {
                "start": "2025-03-20",
                "end": "2030-03-20",
                "frequency": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "bdc": "following",
                "calendar_id": "usny",
                "day_count": "Act360",
                "spread_bp": 100.0,
                "discount_curve_id": "USD-OIS",
            },
            "protection": {
                "credit_curve_id": "CORP-HAZARD",
                "recovery_rate": 0.4,
                "settlement_delay": 3,
            },
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

try:
    validate_instrument_json(cds)
    print("CDS JSON OK")
except Exception as exc:
    print("validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
hazard = HazardCurve(
    "CORP-HAZARD",
    AS_OF,
    [(0.5, 0.018), (1.0, 0.020), (3.0, 0.024), (5.0, 0.028), (10.0, 0.032)],
    recovery_rate=0.40,
)
mc = MarketContext().insert(ois).insert(hazard)
market_json = mc.to_json()
print("hazard id CORP-HAZARD inserted")


## Pricing


In [ ]:
try:
    out = price_instrument_with_metrics(cds, market_json, AS_OF_STR, model="hazard_rate")
    print(ValuationResult.from_json(out))
except Exception as exc:
    print("price:", type(exc).__name__, exc)


## Metrics


In [ ]:
try:
    out = price_instrument_with_metrics(
        cds, market_json, AS_OF_STR, model="hazard_rate", metrics=["cs01", "cs01_hazard", "jump_to_default"]
    )
    vr = ValuationResult.from_json(out)
    for m in ("cs01", "cs01_hazard", "jump_to_default"):
        v = vr.get_metric(m)
        if v is not None:
            print(m, ":", v)
except Exception as exc:
    print("metrics:", type(exc).__name__, exc)
print("Survival: decreasing function of integrated hazard; CS01 links PV to credit spread bumps.")


## Takeaways

- **Model key `hazard_rate`** is the CDS default in the Python bindings.
- **`CORP-HAZARD`** (hazard) and **`USD-OIS`** (discount) must both exist in the market JSON.
- Credit metrics vary by registry wiring; missing metrics are omitted, not errored.
